# Notebook 12: Fine-tuning GPT-2 for Text Generation

**Module 4 · Text Generation**  
*Author: Axel Sirota, Data Trainers LLC*

## The Scenario

In Notebook 11 you trained an LSTM language model on Airbnb rental descriptions. It works, but beyond roughly 30 tokens the output drifts. The reason is structural: the LSTM hidden state is a fixed-size bottleneck, so token 50's memory of token 1 has been squeezed through 49 weight matrices and is severely diluted. Meanwhile you saw BERT outperform the MLP on classification back in Notebook 9. The natural next question: can a transformer *decoder* beat the LSTM at *generation*?

The plan is straightforward. Fine-tune `gpt2` (124M parameters) on the exact same Airbnb dataset you used in Notebook 11, using `AutoModelForCausalLM` and `AutoTokenizer` from HuggingFace together with the standard `Trainer` API for supervised fine-tuning. Then generate text with `model.generate()` using greedy, beam, top-k, top-p, and temperature decoding, and compare output quality, coherence, perplexity, and model size against the LSTM baseline. By the end you will have a fine-tuned GPT-2 that produces coherent 100+ token rental descriptions and a reusable template for fine-tuning any causal LM.

## Learning objectives

By the end of this notebook you will be able to:

1. Contrast encoder-only (BERT) vs. decoder-only (GPT-2) transformers and explain when to use each.
2. Explain causal attention masking: why decoder models only attend to previous tokens.
3. Use `AutoModelForCausalLM` + `AutoTokenizer` to load and tokenize with GPT-2.
4. Prepare causal-LM training data (labels = input_ids shifted by 1, done automatically by HF).
5. Fine-tune GPT-2 with `Trainer` using `DataCollatorForLanguageModeling(mlm=False)`.
6. Generate text with `model.generate(...)` using greedy, beam search, top-k, top-p, and temperature.
7. Compute and compare perplexity for GPT-2 vs. LSTM on held-out Airbnb data.
8. Use `pipeline('text-generation', ...)` for zero-effort inference at deployment.
9. Know the trade-offs: GPT-2 needs 124M params + GPU; LSTM needs ~3M and runs anywhere.

## Prerequisites

- Notebook 11 (LSTM Language Model): you understand the generation task and dataset.
- Notebook 9 (BERT fine-tuning via `Trainer`): you know the HuggingFace training API.
- Comfortable with PyTorch `nn.Module`, `DataLoader`, and `torch.no_grad()`.

## Runtime

Recommended: **Colab with GPU** (T4 is sufficient). Full fine-tuning takes about 45 min on T4. Without GPU, training will be prohibitively slow, so reduce `NUM_EPOCHS` to 1 and `TRAIN_LIMIT` to 2000.

## Section 0. Environment Setup

Install HuggingFace Transformers, Datasets, Accelerate, and Evaluate. On a fresh Colab session this takes about 3 minutes the first time. The model download (124 MB for `gpt2`) happens the first time you call `from_pretrained`.

> **GPU check:** Click *Runtime -> Change runtime type -> T4 GPU* before running this notebook. Training on CPU is roughly 20x slower.

In [ ]:
# Install required packages (run this first in Google Colab).
# If running locally with your virtual environment, you may skip this cell.
!pip install -q \
    "transformers>=4.40" \
    "datasets>=2.19" \
    "accelerate>=0.28" \
    "evaluate>=0.4" \
    "torch>=2.1" \
    "pandas>=2.0" \
    "matplotlib>=3.7"
print("Installation complete.")

In [ ]:
# Imports (visualization, data, model, training).
import os
import math
import time
import random
import warnings
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline,
)
import datasets as hf_datasets

warnings.filterwarnings("ignore")

# --- Reproducibility: seed every RNG we touch ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Device detection ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version      : {torch.__version__}")
print(f"Transformers version : {transformers.__version__}")
print(f"Using device         : {device}")
if device.type == 'cuda':
    print(f"GPU                  : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nEnvironment ready.")

In [ ]:
# Hyperparameters (defined once, reused everywhere).
# Change these only if you need to adapt to memory constraints.

SEED           = 42
MODEL_NAME     = 'gpt2'       # 124M params, ~500MB on disk
MAX_LENGTH     = 128          # Airbnb descriptions are typically 50-150 tokens
NUM_EPOCHS     = 3            # Reduce to 1 on CPU / low-memory GPU
BATCH_SIZE     = 4            # Per-device batch; use GRAD_ACCUM to effectively increase
GRAD_ACCUM     = 4            # Effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LR             = 5e-5         # Standard for GPT-2 fine-tuning
WARMUP_STEPS   = 100          # Linear LR warmup, helps stability
FP16           = torch.cuda.is_available()  # Mixed precision on GPU only
TRAIN_LIMIT    = 5000         # Subsample training rows for speed (use None for all)
TEST_LIMIT     = 500          # Subsample test rows for perplexity evaluation

# Generation hyperparameters
GEN_MAX_NEW    = 40           # Max new tokens beyond the prompt
GEN_TEMP       = 0.8          # Temperature: 1.0 = raw distribution, <1 = sharper
GEN_TOP_K      = 50           # Top-K: only sample from top 50 tokens each step
GEN_TOP_P      = 0.9          # Top-P (nucleus): keep smallest set with cumprob >= 0.9

print(f"Model           : {MODEL_NAME}")
print(f"Max length      : {MAX_LENGTH} tokens")
print(f"Epochs          : {NUM_EPOCHS}")
print(f"Effective batch : {BATCH_SIZE * GRAD_ACCUM}")
print(f"FP16            : {FP16}")

## What We Build Here

Imagine you are a product engineer at Airbnb. A new host uploads a property but writes only three words in the description box: *"Cozy downtown loft"*. The product spec asks for a full 100-word description for review before publication.

The NLP team already tried an LSTM (Notebook 11); it works but drifts after ~30 tokens. Can a transformer do better?

**Pipeline for this notebook:**

```
Raw Airbnb descriptions
        |
        v
  GPT-2 tokenizer (BPE, 50K vocab)
        |
        v
  HF Dataset + DataCollator
        |
        v
  Trainer.train()  <- fine-tune GPT-2 (124M)
        |
        v
  model.generate()  <- greedy / beam / top-k / top-p
        |
        v
  Perplexity comparison: GPT-2 vs LSTM
```

Before any fine-tuning, we first look at what out-of-the-box GPT-2 produces on the same prompts.

## Section 1. Why Transformers for Generation?

### The LSTM ceiling

The LSTM carries meaning from step to step via a hidden state vector of fixed size (e.g. 256 dimensions). Every token has to "fit" through that bottleneck:

```
token_1 -> [h1] -> [h2] -> [h3] -> ... -> [h50] -> token_51
```

By step 50, how much of step 1's meaning survives in `h50`? Mathematically, it's been multiplied by 49 weight matrices. Signal from the distant past is severely attenuated.

### The transformer decoder solution

A transformer decoder (GPT-2) replaces the hidden state with **self-attention**: at every step, it attends directly to every previous token:

```
token_50 attends to [token_1, token_2, ..., token_49] directly
                     no bottleneck, full memory of the past
```

This is enabled by **causal masking**: the attention weight from token `i` to token `j > i` is set to `-inf` before softmax, so the model can only look backward, never forward. This keeps the model autoregressive (it can't cheat by reading the answer).

### GPT-2 model family

| Model | Parameters | Size | Notes |
|---|---|---|---|
| `gpt2` (small) | 124M | ~500MB | We use this. Fast to fine-tune on Colab. |
| `gpt2-medium` | 355M | ~1.4GB | Higher quality, needs more VRAM. |
| `gpt2-large` | 774M | ~3GB | Best quality in the family. |
| `gpt2-xl` | 1.5B | ~6GB | Rarely fits on free Colab. |

For a course environment with a T4 GPU (16GB), `gpt2` (small) is the right choice.

## Section 2. Using Pretrained GPT-2 Out of the Box

Before any fine-tuning, let's explore the pretrained model. This gives us a quality baseline to compare against after training.

In [ ]:
# Load GPT-2 tokenizer.
# IMPORTANT: GPT-2 has no padding token by default, so we must set one.
# The standard workaround is to reuse the EOS token as PAD.
print(f"Loading tokenizer for '{MODEL_NAME}' ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Without this line, batched tokenization will crash.
tokenizer.pad_token = tokenizer.eos_token  # <-- mandatory GPT-2 workaround

print(f"Vocab size        : {tokenizer.vocab_size:,}")
print(f"Max model length  : {tokenizer.model_max_length}")
print(f"EOS token         : {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")
print(f"PAD token         : {tokenizer.pad_token!r}  (id={tokenizer.pad_token_id})")
print(f"BOS token         : {tokenizer.bos_token!r}  (id={tokenizer.bos_token_id})")

In [ ]:
# See how GPT-2 tokenizes a sample rental description.
# GPT-2 uses Byte-Pair Encoding (BPE): common subwords become single tokens.

sample_text = "This cozy downtown loft features exposed brick and modern amenities."
tokens = tokenizer.tokenize(sample_text)  # raw subword strings
ids    = tokenizer.encode(sample_text)    # integer token IDs

print(f"Original text  : {sample_text}")
print(f"Tokens ({len(tokens)})     : {tokens}")
print(f"Token IDs      : {ids}")
print()
# Contrast: LSTM from NB11 used a word-level vocab of ~5K words.
# GPT-2 BPE has 50,257 subwords, so it handles any word, including OOV.
print("Note: BPE splits 'amenities' into subword pieces, so no OOV tokens.")

In [ ]:
# Load base GPT-2 model (no fine-tuning yet).
print(f"Loading '{MODEL_NAME}' model (~500MB, may take a moment) ...")
t0 = time.time()
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
base_model = base_model.to(device)
base_model.eval()  # inference mode

n_params = sum(p.numel() for p in base_model.parameters())
print(f"Loaded in {time.time() - t0:.1f}s")
print(f"Parameters    : {n_params:,}  (~{n_params/1e6:.0f}M)")
print(f"Device        : {next(base_model.parameters()).device}")

In [ ]:
# Quick demo: pretrained GPT-2 on a generic prompt.
# This shows what the model produces WITHOUT any Airbnb fine-tuning.

generic_pipe = pipeline(
    'text-generation',
    model=base_model,
    tokenizer=tokenizer,
    device=0 if device.type == 'cuda' else -1,  # 0 = first GPU, -1 = CPU
)

prompt_generic = "The capital of France is"
out = generic_pipe(
    prompt_generic,
    max_new_tokens=30,
    do_sample=False,          # greedy for determinism
    pad_token_id=tokenizer.eos_token_id,
)[0]['generated_text']
print(f"Prompt  : {prompt_generic!r}")
print(f"Output  : {out}")

In [ ]:
# Now try an Airbnb-style prompt.
# Pretrained GPT-2 has never seen Airbnb listings, so output will be generic.

prompt_airbnb = "This rental offers"
out = generic_pipe(
    prompt_airbnb,
    max_new_tokens=40,
    do_sample=True,
    temperature=GEN_TEMP,
    top_k=GEN_TOP_K,
    top_p=GEN_TOP_P,
    pad_token_id=tokenizer.eos_token_id,
)[0]['generated_text']
print(f"Prompt  : {prompt_airbnb!r}")
print(f"Output  : {out}")
print()
print("The text is grammatically OK but NOT Airbnb-style.")
print("After fine-tuning it will match the domain vocabulary.")

### Demo: Decoding strategies with the base model

Before fine-tuning, let's explore the five main decoding strategies using `model.generate()` directly. This is the low-level API that `pipeline` wraps.

In [ ]:
# Helper: tokenize a prompt and generate with model.generate().
# We will reuse this throughout the notebook.

def generate_text(model, prompt, max_new_tokens=GEN_MAX_NEW, **gen_kwargs):
    """Tokenize `prompt`, generate tokens, decode and return the full string."""
    # Tokenize prompt and move to device
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,   # required for GPT-2
            eos_token_id=tokenizer.eos_token_id,   # stop at EOS
            **gen_kwargs,
        )

    # Decode and strip special tokens
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# Five decoding strategies on the same prompt
prompt = "This rental offers stunning views of"

strategies = [
    ("Greedy",          dict(do_sample=False)),
    ("Beam search (4)", dict(do_sample=False, num_beams=4, early_stopping=True)),
    ("Temp=0.5",        dict(do_sample=True, temperature=0.5)),
    ("Top-K=50",        dict(do_sample=True, top_k=50)),
    ("Top-P=0.9",       dict(do_sample=True, top_p=0.9, top_k=0)),
]

print(f"Prompt: {prompt!r}\n")
for name, kwargs in strategies:
    text = generate_text(base_model, prompt, **kwargs)
    continuation = text[len(prompt):].strip()
    print(f"[{name:<17}] {continuation[:120]}")

### Lab 1. Explore Decoding Strategies

Try the base model (before fine-tuning) with different decoding strategies and observe the output.

**Steps:**

1. Pick a prompt (e.g. `"The apartment is located"`).
2. Generate 3 samples using:
   - Greedy (`do_sample=False`)
   - Temperature=1.0 (random sampling with raw distribution)
   - Top-P=0.95 (nucleus sampling)
3. Print the 3 outputs side by side.
4. In a comment, note: which one seems most coherent? Most creative? Most repetitive?

In [ ]:
# Lab 1: decoding strategies exploration.

# 1. Choose a prompt
lab_prompt = None  # YOUR CODE  (e.g. "The apartment is located")

# 2a. Greedy decoding (deterministic, always picks most likely next token)
greedy_output = None  # YOUR CODE  (hint: generate_text(base_model, lab_prompt, do_sample=False))

# 2b. Temperature=1.0 (sample from raw distribution, more diverse)
temp_output = None    # YOUR CODE  (hint: generate_text(..., do_sample=True, temperature=1.0))

# 2c. Top-P=0.95 nucleus sampling (keep smallest set of tokens with cumprob >= 0.95)
topp_output = None    # YOUR CODE  (hint: generate_text(..., do_sample=True, top_p=0.95, top_k=0))

# 3. Print results
if greedy_output is not None:
    print(f"Prompt : {lab_prompt!r}\n")
    print(f"[Greedy]    : {greedy_output[len(lab_prompt):].strip()[:150]}")
    print(f"[Temp=1.0]  : {temp_output[len(lab_prompt):].strip()[:150]}")
    print(f"[Top-P=0.95]: {topp_output[len(lab_prompt):].strip()[:150]}")

# 4. Your observation as a comment:
# Greedy tends to be: ...
# Temperature=1.0 tends to be: ...
# Top-P tends to be: ...

## Section 3. Data Preparation for Causal LM Fine-tuning

### Causal LM format

For a standard language model, each training sample is just a sequence of tokens. The model predicts the next token at every position. HuggingFace's `DataCollatorForLanguageModeling(mlm=False)` automatically constructs:

```
input_ids : [t0, t1, t2, t3, t4]
labels    : [t1, t2, t3, t4, t5]   <- shifted by 1
```

The `<|endoftext|>` token acts as both BOS (start) and EOS (end) in GPT-2. We prefix each description with it so the model learns to start and stop cleanly.

### Why `mlm=False`?

`DataCollatorForLanguageModeling` has a parameter `mlm` (masked language modeling). Setting `mlm=True` is for BERT-style training (randomly mask tokens, predict masked). Setting `mlm=False` is for GPT-style **causal** LM training (predict next token, no masking). **Always use `mlm=False` for GPT-2.**

In [ ]:
# Download the Airbnb descriptions dataset (same train/test split as Notebook 11).
TRAIN_URL = "https://www.dropbox.com/scl/fi/rbrynlq7871cshi0krftj/train_corpus_descriptions_airbnb.csv?rlkey=td1pfjgqjccap0xu9g4eliube&dl=1"
TEST_URL  = "https://www.dropbox.com/scl/fi/eys05bzwwnhskadqh7aux/test_corpus_descriptions_airbnb.csv?rlkey=p1zuz90khh5t7dx3hkfba1dzm&dl=1"

TRAIN_PATH = "./airbnb_train.csv"
TEST_PATH  = "./airbnb_test.csv"

if not os.path.exists(TRAIN_PATH):
    print("Downloading Airbnb train set ...")
    urllib.request.urlretrieve(TRAIN_URL, TRAIN_PATH)

if not os.path.exists(TEST_PATH):
    print("Downloading Airbnb test set ...")
    urllib.request.urlretrieve(TEST_URL, TEST_PATH)

# Load and inspect
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

# The description text column may vary in name; find it
text_col = [c for c in train_df.columns if 'descrip' in c.lower() or 'text' in c.lower()][0]
print(f"Text column: {text_col!r}")

# Clean and subsample
train_texts = train_df[text_col].dropna().astype(str).str.strip().tolist()
test_texts  = test_df[text_col].dropna().astype(str).str.strip().tolist()

if TRAIN_LIMIT:
    random.shuffle(train_texts)
    train_texts = train_texts[:TRAIN_LIMIT]
if TEST_LIMIT:
    test_texts = test_texts[:TEST_LIMIT]

print(f"\nTrain descriptions : {len(train_texts):,}")
print(f"Test descriptions  : {len(test_texts):,}")
print(f"\nSample description:\n{train_texts[0][:300]}")

In [ ]:
# Format each description as:  <|endoftext|> + description
# GPT-2 uses <|endoftext|> as both BOS and EOS (standard convention).
BOS = tokenizer.bos_token  # '<|endoftext|>'

train_formatted = [f"{BOS}{text}" for text in train_texts]
test_formatted  = [f"{BOS}{text}" for text in test_texts]

print(f"BOS token: {BOS!r}")
print(f"Formatted train[0]:\n{train_formatted[0][:200]}\n")

# Quick token length analysis to choose MAX_LENGTH
sample_lens = [
    len(tokenizer.encode(t, truncation=False)) for t in train_formatted[:500]
]
print(f"Token length stats (first 500 samples):")
print(f"  mean  = {np.mean(sample_lens):.0f}")
print(f"  median = {np.median(sample_lens):.0f}")
print(f"  p90    = {np.percentile(sample_lens, 90):.0f}")
print(f"  max    = {max(sample_lens)}")
print(f"  Our MAX_LENGTH = {MAX_LENGTH}  (truncates ~{sum(l > MAX_LENGTH for l in sample_lens)/len(sample_lens)*100:.0f}% of samples)")

In [ ]:
# Tokenize all texts and convert to HuggingFace Dataset objects.
# truncation=True and max_length=MAX_LENGTH: longer descriptions are cut.

def tokenize_batch(texts, max_length=MAX_LENGTH):
    """Tokenize a list of texts and return a HuggingFace Dataset."""
    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=max_length,
        padding=False,           # DataCollator will handle padding dynamically
        return_overflowing_tokens=False,
    )
    # Build a plain dict then wrap in HF Dataset
    data = {'input_ids': encodings['input_ids'], 'attention_mask': encodings['attention_mask']}
    return hf_datasets.Dataset.from_dict(data)

print("Tokenizing train set ...")
train_dataset = tokenize_batch(train_formatted)
print("Tokenizing test set ...")
test_dataset  = tokenize_batch(test_formatted)

print(f"\nTrain dataset : {train_dataset}")
print(f"Test dataset  : {test_dataset}")
print(f"\nFirst sample token IDs: {train_dataset[0]['input_ids'][:20]} ...")

In [ ]:
# DataCollator for causal LM.
# mlm=False  ->  don't randomly mask tokens (that's BERT)
# It will: pad sequences to equal length within each batch,
#          set labels = input_ids.clone(), and set label=-100 on padding positions
#          so CrossEntropyLoss ignores them.

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,   # CRITICAL: False = causal LM (GPT-2); True = masked LM (BERT)
)

# Quick sanity check: collate two samples and inspect shapes
sample_batch = [train_dataset[i] for i in range(2)]
collated = data_collator(sample_batch)
print("DataCollator output for 2 samples:")
for k, v in collated.items():
    print(f"  {k:<20} shape={tuple(v.shape)}  dtype={v.dtype}")
print()
print("labels=-100 at padding positions (ignored by loss):")
print(collated['labels'][0][:30])

### Lab 2. Prepare the Dataset Pipeline

You have seen how `tokenize_batch` and `DataCollatorForLanguageModeling` work. Now set up the pipeline from scratch on a small sample to make sure you understand every step.

**Steps:**

1. Take the first 20 descriptions from `train_texts`. Format them with `<|endoftext|>` prefix.
2. Tokenize them using the tokenizer (with `truncation=True, max_length=64`).
3. Create a `DataCollatorForLanguageModeling(tokenizer, mlm=False)` and collate your 20 samples into a batch.
4. Print the shape of `input_ids`, `attention_mask`, and `labels` tensors.
5. Verify that `labels` has `-100` entries wherever `attention_mask == 0`.

In [ ]:
# Lab 2: dataset pipeline from scratch.

# 1. Take 20 descriptions and format with BOS
mini_texts = None  # YOUR CODE  (hint: list of first 20 from train_texts, with BOS prefix)

# 2. Tokenize with truncation
mini_dataset = None  # YOUR CODE  (hint: tokenize_batch(mini_texts, max_length=64))

# 3. Create collator (mlm=False!)
mini_collator = None  # YOUR CODE  (hint: DataCollatorForLanguageModeling(tokenizer, mlm=False))

# 4. Collate all 20 samples into a single batch
mini_batch = None  # YOUR CODE  (hint: mini_collator([mini_dataset[i] for i in range(20)]))

# Print shapes
if mini_batch is not None:
    for k, v in mini_batch.items():
        print(f"{k:<20} shape={tuple(v.shape)}")

    # 5. Verify -100 on padding positions
    mask   = mini_batch['attention_mask']
    labels = mini_batch['labels']
    n_ignored = (labels == -100).sum().item()
    n_pad     = (mask == 0).sum().item()
    print(f"\nPadding positions (mask==0): {n_pad}")
    print(f"Ignored label positions (-100): {n_ignored}")
    print(f"Match: {n_ignored >= n_pad}  (ignored >= pad is expected)")

## Section 4. Fine-tuning with the HuggingFace Trainer

The `Trainer` handles the training loop, gradient accumulation, mixed precision, logging, and checkpointing. We only need to specify `TrainingArguments` and hand it the datasets + collator.

**Key arguments:**

| Argument | Value | Why |
|---|---|---|
| `num_train_epochs` | 3 | 1-3 epochs avoids overfitting on small corpora |
| `per_device_train_batch_size` | 4 | Fits T4 memory |
| `gradient_accumulation_steps` | 4 | Effective batch = 16 with less peak memory |
| `learning_rate` | 5e-5 | Standard for GPT-2 SFT; don't go higher |
| `warmup_steps` | 100 | Linear LR ramp-up prevents early divergence |
| `fp16` | True | Half-precision on GPU; 2x memory saving |
| `logging_steps` | 50 | Print loss every 50 steps |

In [ ]:
# Load a fresh copy of GPT-2 for fine-tuning (separate from base_model).
print(f"Loading '{MODEL_NAME}' for fine-tuning ...")
ft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
ft_model = ft_model.to(device)

# Resize embeddings if we added tokens (we didn't, but good habit).
ft_model.resize_token_embeddings(len(tokenizer))

print(f"Model ready. Parameters: {sum(p.numel() for p in ft_model.parameters()):,}")

In [ ]:
# TrainingArguments: all training config in one place.
training_args = TrainingArguments(
    output_dir='./gpt2-airbnb',              # where to save checkpoints
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,  # effective batch = BATCH_SIZE * GRAD_ACCUM
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    fp16=FP16,                               # half-precision if GPU available
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',                   # evaluate at end of each epoch
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,                 # lower loss = better
    report_to='none',                        # disable wandb / tensorboard
    seed=SEED,
)

print("TrainingArguments configured.")
print(f"  Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  FP16                 : {FP16}")
print(f"  Output dir           : {training_args.output_dir}")

In [ ]:
# Build the Trainer and start training.
# Expected: ~15-20 min on Colab T4 with TRAIN_LIMIT=5000, NUM_EPOCHS=3.

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting fine-tuning ...")
t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0

print(f"\nTraining complete in {elapsed/60:.1f} minutes.")
print(f"Final training loss : {train_result.training_loss:.4f}")

In [ ]:
# Evaluate on the held-out test set and compute perplexity.
# Perplexity = exp(cross-entropy loss)  -- lower is better.

eval_results = trainer.evaluate()
eval_loss = eval_results['eval_loss']
perplexity = math.exp(eval_loss)

print(f"Test eval_loss  : {eval_loss:.4f}")
print(f"Test perplexity : {perplexity:.2f}")
print()
print("Reference: Notebook 11 LSTM perplexity ~100 (word-level)")
print("Expected GPT-2 fine-tuned perplexity    : ~20-40 (BPE, different tokenizer!)")
print()
print("IMPORTANT CAVEAT: These perplexity numbers are NOT directly comparable.")
print("LSTM uses a ~5K word vocabulary; GPT-2 uses 50K BPE subwords.")
print("Different vocabularies = different token counts = incomparable perplexity.")
print("Use qualitative comparison (sample quality) for a fair judgment.")

In [ ]:
# Save the fine-tuned model to disk.
SAVE_DIR = './gpt2-airbnb-final'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Fine-tuned model saved to: {SAVE_DIR}")

# Verify we can reload it and get identical output.
reloaded = AutoModelForCausalLM.from_pretrained(SAVE_DIR).to(device)
reloaded.eval()

# Quick smoke test: compare one generation from ft_model vs reloaded
ft_model.eval()
test_prompt = "This spacious apartment features"
out_orig  = generate_text(ft_model, test_prompt, do_sample=False)
out_reload = generate_text(reloaded,  test_prompt, do_sample=False)
print(f"Original  : {out_orig[:150]}")
print(f"Reloaded  : {out_reload[:150]}")
print(f"Match: {out_orig == out_reload}")

### Lab 3. Retrain with Fewer Epochs

Compare perplexity and output quality when fine-tuning for 1 epoch vs. 3 epochs.

**Steps:**

1. Create a new `TrainingArguments` with `num_train_epochs=1` (keep everything else the same).
2. Create a new `Trainer` with a fresh `AutoModelForCausalLM.from_pretrained(MODEL_NAME)` model.
3. Train for 1 epoch.
4. Evaluate and print `eval_loss` + perplexity.
5. Generate 3 samples with both the 1-epoch model and the 3-epoch model on the same prompt.
6. Comment: does quality improve significantly from epoch 1 to epoch 3?

In [ ]:
# Lab 3: 1-epoch vs 3-epoch comparison.

# 1. TrainingArguments for 1 epoch
args_1ep = None  # YOUR CODE  (hint: TrainingArguments(..., num_train_epochs=1, ...))

# 2. Fresh model
model_1ep = None  # YOUR CODE  (hint: AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device))

# 3. Trainer
trainer_1ep = None  # YOUR CODE

# 4. Train + evaluate
if trainer_1ep is not None:
    trainer_1ep.train()
    res_1ep = trainer_1ep.evaluate()
    ppl_1ep = math.exp(res_1ep['eval_loss'])
    print(f"1-epoch perplexity : {ppl_1ep:.2f}")
    print(f"3-epoch perplexity : {perplexity:.2f}")

    # 5. Side-by-side samples
    model_1ep.eval()
    cmp_prompt = "The bedroom has"
    for m, label in [(model_1ep, "1-epoch"), (ft_model, "3-epoch")]:
        text = generate_text(m, cmp_prompt, do_sample=True, temperature=0.8, top_k=50)
        print(f"[{label}] {text[len(cmp_prompt):].strip()[:150]}")

## Section 5. Decoding Strategies Deep Dive

Now that we have a fine-tuned model, let's do a proper comparison of all five decoding strategies side by side. The same strategy that sounded okay with the base model may produce very different results with the fine-tuned Airbnb model.

### Greedy decoding

At each step, pick the single token with the highest probability:

```python
next_token = argmax(logits)
```

**Pros:** Deterministic, reproducible, simple.  
**Cons:** Tends to repeat phrases ("the the the..."), lacks diversity, often sub-optimal globally.

### Beam search

Maintain the top-B partial sequences ("beams") at each step. Choose the beam with highest cumulative log-probability at the end:

```python
model.generate(..., num_beams=4, early_stopping=True)
```

**Pros:** Better than greedy, still deterministic.  
**Cons:** Can produce safe, repetitive, and boring text (especially for open-ended generation).

### Temperature sampling

Divide the logits by `T` before softmax. `T < 1` sharpens the distribution (less diverse), `T > 1` flattens it (more random):

```python
probs = softmax(logits / T)
next_token = multinomial_sample(probs)
```

### Top-K sampling

Zero out all but the top-K logits before sampling:

```python
top_k_logits, top_k_indices = torch.topk(logits, k=50)
probs = softmax(top_k_logits)  # redistributed over top-50 only
```

**Pros:** Avoids obvious garbage tokens.  
**Cons:** K is fixed; if the top-K includes too many similar tokens, the output is still repetitive.

### Top-P (nucleus) sampling

Keep the **smallest set** of tokens whose cumulative probability sum >= P:

```python
sorted_probs = sort_descending(softmax(logits))
cumsum = cumulative_sum(sorted_probs)
nucleus = sorted_probs[cumsum <= p]   # dynamic k, different each step
```

**Pros:** Adaptive. When the model is confident (one token dominates), the nucleus is small; when uncertain, it's large.  
**Default recommendation:** `top_p=0.9` with `temperature=0.8` is a good starting point.

### Repetition penalty

An additional knob: divide the logits of already-generated tokens by `repetition_penalty > 1`:

```python
model.generate(..., repetition_penalty=1.2)
```

Useful when the model still repeats despite top-K / top-P.

In [ ]:
# Five decoding strategies on the FINE-TUNED model, all five Airbnb prompts.
ft_model.eval()

test_prompts = [
    "This rental offers stunning views of",
    "The kitchen features",
    "Walking distance to",
    "Perfect for couples,",
    "The neighborhood is",
]

strategies_ft = [
    ("Greedy",              dict(do_sample=False)),
    ("Beam (k=4)",          dict(do_sample=False, num_beams=4, early_stopping=True)),
    ("Temp=0.8",            dict(do_sample=True, temperature=0.8)),
    ("Top-K=50",            dict(do_sample=True, top_k=50)),
    ("Top-P=0.9, T=0.8",    dict(do_sample=True, top_p=0.9, top_k=0, temperature=0.8)),
]

# Print a table: each prompt x each strategy
for prompt in test_prompts:
    print(f"\n{'='*70}")
    print(f"PROMPT: {prompt!r}")
    print('='*70)
    for name, kwargs in strategies_ft:
        text = generate_text(ft_model, prompt, **kwargs)
        cont = text[len(prompt):].strip()[:100]
        print(f"  [{name:<20}] {cont}")

### Lab 4. Your Best Airbnb Description

Use the fine-tuned model to generate the best Airbnb-style description you can.

**Steps:**

1. Choose a prompt (think of a real rental scenario).
2. Experiment with at least 3 different decoding strategies or hyperparameter settings.
3. Print all 3 outputs.
4. Pick your favorite and explain why in a comment (2-3 sentences).

In [ ]:
# Lab 4: best Airbnb description.

# 1. Your prompt
my_prompt = None  # YOUR CODE  (e.g. "Charming beach cottage with")

# 2a. Strategy 1: your choice
output_a = None  # YOUR CODE

# 2b. Strategy 2: your choice
output_b = None  # YOUR CODE

# 2c. Strategy 3: try repetition_penalty=1.2
output_c = None  # YOUR CODE  (hint: generate_text(ft_model, my_prompt, do_sample=True, top_p=0.9, temperature=0.8, repetition_penalty=1.2))

# 3. Print all
if my_prompt is not None:
    for label, out in [("Strategy A", output_a), ("Strategy B", output_b), ("Strategy C (rep_pen)", output_c)]:
        print(f"--- {label} ---")
        if out:
            print(out)
        print()

# 4. Your comment: which is best and why?
# My favorite is: ...
# Because: ...

## Section 6. Head-to-Head: GPT-2 vs. LSTM

Now let's compare GPT-2 with the LSTM from Notebook 11 on the same prompts. If you didn't run Notebook 11, we include reference outputs below.

In [ ]:
# Reference LSTM outputs (from a completed Notebook 11 run).
# If you ran Notebook 11, load your saved model instead.
# These are representative samples for illustration.

lstm_reference_outputs = {
    "This rental offers stunning views of": 
        "the city skyline from the private balcony . the apartment is located in the heart of . "
        "the apartment is a great place to stay in . the apartment is clean and ",
    "The kitchen features":
        "a fully equipped kitchen with stainless steel appliances . the kitchen is equipped with a "
        "full kitchen with all the amenities you need . the kitchen is perfect for",
    "Walking distance to":
        "the beach and local restaurants . the apartment is located in a quiet neighborhood . "
        "the apartment is a short walk from the subway . the apartment is a",
}

# GPT-2 fine-tuned on the same prompts for comparison.
comparison_prompts = list(lstm_reference_outputs.keys())
print("=" * 70)
print("HEAD-TO-HEAD: GPT-2 fine-tuned vs LSTM reference outputs")
print("=" * 70)

for prompt in comparison_prompts:
    gpt2_out = generate_text(
        ft_model, prompt,
        do_sample=True, temperature=0.8, top_k=50, top_p=0.9
    )
    gpt2_cont  = gpt2_out[len(prompt):].strip()[:200]
    lstm_cont  = lstm_reference_outputs[prompt]

    print(f"\nPROMPT: {prompt!r}")
    print(f"  LSTM   : {lstm_cont[:150]}")
    print(f"  GPT-2  : {gpt2_cont[:150]}")

In [ ]:
# Quantitative comparison: size, parameter count, training data efficiency.
import shutil

gpt2_size_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(SAVE_DIR) if os.path.exists(SAVE_DIR)
    for f in files
) / 1e6

print("=" * 50)
print("QUANTITATIVE COMPARISON")
print("=" * 50)
print(f"{'Metric':<30} {'LSTM (NB11)':>15} {'GPT-2 (NB12)':>15}")
print("-" * 62)
print(f"{'Parameters':.<30} {'~3M':>15} {'124M':>15}")
print(f"{'Model size (disk)':.<30} {'~15 MB':>15} {f'{gpt2_size_mb:.0f} MB':>15}")
print(f"{'Vocab size':.<30} {'~5K (word)':>15} {'50K (BPE)':>15}")
print(f"{'Test perplexity':.<30} {'~100 (word)':>15} {f'{perplexity:.1f} (BPE)':>15}")
print(f"{'Coherent beyond 30 tokens':.<30} {'Rarely':>15} {'Often':>15}")
print(f"{'Runs on CPU (fast)':.<30} {'Yes':>15} {'Slow':>15}")
print(f"{'OOV handling':.<30} {'<UNK>':>15} {'Subword split':>15}")
print()
print("NOTE: Perplexity numbers use different tokenizers, so not directly comparable.")
print("The qualitative difference (sample quality) is the meaningful comparison.")

In [ ]:
# Inference latency: time to generate 40 tokens on current device.
timing_prompt = "This rental offers stunning views of"
n_runs = 5  # average over multiple runs

ft_model.eval()
times = []
for _ in range(n_runs):
    t0 = time.time()
    _ = generate_text(
        ft_model, timing_prompt,
        max_new_tokens=40,
        do_sample=False,  # greedy for determinism
    )
    times.append(time.time() - t0)

avg_time = np.mean(times)
print(f"GPT-2 inference ({40} tokens) on {device}:")
print(f"  Mean : {avg_time*1000:.0f} ms")
print(f"  Runs : {times}")
print()
print("On CPU this will be ~10-30x slower than GPU.")
print("For production at scale: distilgpt2 (82M) or tinygpt2-based models reduce latency.")

## Section 7. Deployment with `pipeline`

For production inference, HuggingFace's `pipeline` API wraps the tokenize -> generate -> decode loop in a single call. This is what you'd use in a REST API endpoint.

In [ ]:
# Production-style inference with pipeline.
# You can swap in any checkpoint path; the pipeline handles everything.

gen_pipeline = pipeline(
    'text-generation',
    model=ft_model,
    tokenizer=tokenizer,
    device=0 if device.type == 'cuda' else -1,
)

# Generate multiple samples with a single call (num_return_sequences)
results = gen_pipeline(
    "Bright and airy studio apartment",
    max_new_tokens=GEN_MAX_NEW,
    num_return_sequences=3,         # 3 samples
    do_sample=True,
    temperature=GEN_TEMP,
    top_k=GEN_TOP_K,
    top_p=GEN_TOP_P,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
)

print("Pipeline output (3 samples):\n")
for i, r in enumerate(results):
    print(f"Sample {i+1}: {r['generated_text'][:300]}\n")

## Optional Lab. Use the `pipeline` from a Saved Checkpoint

Practice the zero-shot reload workflow that you'd use when deploying a model someone else fine-tuned.

**Steps:**

1. Load the tokenizer and model from `SAVE_DIR` using `AutoTokenizer.from_pretrained(SAVE_DIR)` and `AutoModelForCausalLM.from_pretrained(SAVE_DIR)`.
2. Build a `pipeline('text-generation', ...)` from these loaded objects.
3. Generate 5 samples for the prompt `"The host was very helpful and"` with your favorite decoding strategy.
4. Compare: does the output look like Airbnb-style writing or generic web text?

In [ ]:
# Optional Lab: load from checkpoint and run pipeline.

# 1. Load tokenizer + model from SAVE_DIR
loaded_tokenizer = None  # YOUR CODE  (hint: AutoTokenizer.from_pretrained(SAVE_DIR))
loaded_model     = None  # YOUR CODE  (hint: AutoModelForCausalLM.from_pretrained(SAVE_DIR).to(device))

# 2. Build pipeline
loaded_pipe = None  # YOUR CODE

# 3. Generate
if loaded_pipe is not None:
    outs = loaded_pipe(
        "The host was very helpful and",
        max_new_tokens=40,
        num_return_sequences=5,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=loaded_tokenizer.eos_token_id,
    )
    for i, o in enumerate(outs):
        print(f"Sample {i+1}: {o['generated_text'][:200]}")

## Section 8. Wrap-up

### When LSTM is still the right answer

- **Edge devices** (Raspberry Pi, mobile): 3M params fit in RAM; 124M may not.
- **Tiny corpora** (<500 samples): fine-tuning GPT-2 on 100 sentences will overfit catastrophically; an LSTM with appropriate regularization may generalize better.
- **Real-time low-latency inference on CPU**: an LSTM generates tokens roughly 10x faster on CPU than GPT-2.
- **Interpretability**: LSTM hidden states are sometimes easier to probe than transformer attention.

### When GPT-2 wins

- **Open-ended generation**: long, coherent paragraphs where LSTMs drift.
- **Domain shift via fine-tuning**: starting from a 124M-param pretrained checkpoint instead of random weights gives a massive head start, especially with <10K training examples.
- **BPE tokenization**: handles any word including OOV (no `<UNK>` tokens).
- **Ecosystem**: HuggingFace `pipeline` + `Trainer` + `model.generate()` are industry-standard APIs.

### Where this goes next

GPT-2 is a 2019 model. Current open-source LLMs (`GPT-4`, `Claude`, `Llama-3`, `Mistral`) are all variants of the same decoder-only architecture, just with orders of magnitude more parameters and data. The fine-tuning recipe you learned today applies directly to smaller open-source models such as `TinyLlama`, `Phi-2`, and `Gemma-2B`.

For **Retrieval-Augmented Generation (RAG)**, where you pair generation with a vector search over a document store, see the `chatbots-with-genai` and `genai_and_llms` courses. The sentence embeddings from Notebook 7 are the retrieval component; the GPT-2 fine-tuning recipe here is the generation component.

### Self-check quiz

1. Why do we set `tokenizer.pad_token = tokenizer.eos_token` for GPT-2? What error would occur without it?
2. You set `DataCollatorForLanguageModeling(mlm=True)` by mistake. Describe what goes wrong during training.
3. Your fine-tuned GPT-2 test perplexity is **higher** than the base (un-fine-tuned) GPT-2. Name 3 likely bugs or causes.
4. Greedy decoding produced `"bedroom bedroom bedroom..."`. Top-k=50 didn't help. What else could you try?
5. Name a scenario where an LSTM-based LM is preferable to GPT-2 fine-tuned.

Notebook 12 complete. Notebook 13 is the capstone: Shakespeare generation with LSTM vs. GPT-2.